In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from patsy import dmatrix
import matplotlib.pyplot as plt
from pathlib import Path

#DATA_PATH = Path("data/processed/demand_country_hourly.parquet")

# Country -> IANA timezone. Extend as needed for your dataset.
COUNTRY_TZ = {
    "AT": "Europe/Vienna", "BE": "Europe/Brussels", "BG": "Europe/Sofia",
    "CH": "Europe/Zurich", "CZ": "Europe/Prague", "DE": "Europe/Berlin",
    "DK": "Europe/Copenhagen", "EE": "Europe/Tallinn", "ES": "Europe/Madrid",
    "FI": "Europe/Helsinki", "FR": "Europe/Paris", "GR": "Europe/Athens",
    "HR": "Europe/Zagreb", "HU": "Europe/Budapest", "IE": "Europe/Dublin",
    "IT": "Europe/Rome", "LT": "Europe/Vilnius", "LU": "Europe/Luxembourg",
    "LV": "Europe/Riga", "NL": "Europe/Amsterdam", "NO": "Europe/Oslo",
    "PL": "Europe/Warsaw", "PT": "Europe/Lisbon", "RO": "Europe/Bucharest",
    "RS": "Europe/Belgrade", "SE": "Europe/Stockholm", "SI": "Europe/Ljubljana",
    "SK": "Europe/Bratislava", "GB": "Europe/London", "UK": "Europe/London",
}

In [3]:
df = pd.read_parquet('/Users/ryanaskin/Documents/datasci-challenge/renewables-dataset.parquet')

# Ensure timestamp is tz-aware UTC
df["Time"] = pd.to_datetime(df["Time"], utc=True)
df = df.sort_values(["country", "Time"]).reset_index(drop=True)

print(df.head())
print(f"\nShape: {df.shape}")
print(f"Countries: {sorted(df['country'].unique())}")
print(f"Range: {df['Time'].min()} → {df['Time'].max()}")

                       Time    ID  demand_MWh  supply_MWh  solar_MWh  \
0 2012-01-01 00:00:00+00:00  1277    221.1898    8.203896        0.0   
1 2012-01-01 00:00:00+00:00  1278    290.9656    3.410292        0.0   
2 2012-01-01 00:00:00+00:00  1279     19.3036   17.774111        0.0   
3 2012-01-01 00:00:00+00:00  1301     66.5712   28.651808        0.0   
4 2012-01-01 00:00:00+00:00  1302     41.3080   20.062614        0.0   

    wind_MWh  solar_rel_prod  wind_rel_prod   latitude  longitude country  \
0  16.407792             0.0         0.0363  40.844443  20.266280     ALB   
1   6.820585             0.0         0.0406  41.419464  19.859433     ALB   
2  35.548222             0.0         0.2352  41.702341  19.610791     ALB   
3  57.303615             0.0         0.1164  41.030359  19.159265     ALB   
4  40.125228             0.0         0.1687  41.282682  19.303039     ALB   

   solar_layout_MW  wind_layout_MW  
0        2493.9715        452.0053  
1         927.3392        167.

In [8]:
COUNTRY_TZ = {
    'POR': 'Europe/Lisbon',       'ESP': 'Europe/Madrid',
    'FRA': 'Europe/Paris',        'BEL': 'Europe/Brussels',
    'CHE': 'Europe/Zurich',       'LUX': 'Europe/Luxembourg',
    'NLD': 'Europe/Amsterdam',    'ITA': 'Europe/Rome',
    'DEU': 'Europe/Berlin',       'AUT': 'Europe/Vienna',
    'DNK': 'Europe/Copenhagen',   'CZE': 'Europe/Prague',
    'POL': 'Europe/Warsaw',       'HUN': 'Europe/Budapest',
    'SVK': 'Europe/Bratislava',   'SVN': 'Europe/Ljubljana',
    'HRV': 'Europe/Zagreb',       'GRC': 'Europe/Athens',
    'ALB': 'Europe/Tirane',       'MKD': 'Europe/Skopje',
    'BGR': 'Europe/Sofia',        'MNE': 'Europe/Podgorica',
    'BIH': 'Europe/Sarajevo',     'SRB': 'Europe/Belgrade',
    'ROU': 'Europe/Bucharest',
}

In [9]:
def add_calendar_features(sub: pd.DataFrame, tz: str) -> pd.DataFrame:
    """Add local-time calendar features. `sub` is a single-country slice."""
    local = sub["Time"].dt.tz_convert(tz)
    sub = sub.copy()
    sub["local_hour"] = local.dt.hour
    sub["local_dow"] = local.dt.dayofweek                # 0=Mon
    sub["hour_of_week"] = sub["local_dow"] * 24 + sub["local_hour"]  # 0..167
    sub["month"] = local.dt.month
    sub["log_demand"] = np.log(sub["demand_MWh"])
    return sub

pieces = []
for country, sub in df.groupby("country", sort=False):
    tz = COUNTRY_TZ.get(country, "UTC")
    if tz == "UTC":
        print(f"[warn] no timezone for {country}, using UTC")
    pieces.append(add_calendar_features(sub, tz))
feat = pd.concat(pieces, ignore_index=True)
feat.head()

/opt/homebrew/anaconda3/envs/data-science-challenge/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/homebrew/anaconda3/envs/data-science-challenge/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/homebrew/anaconda3/envs/data-science-challenge/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/homebrew/anaconda3/envs/data-science-challenge/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,Time,ID,demand_MWh,supply_MWh,solar_MWh,wind_MWh,solar_rel_prod,wind_rel_prod,latitude,longitude,country,solar_layout_MW,wind_layout_MW,local_hour,local_dow,hour_of_week,month,log_demand
0,2012-01-01 00:00:00+00:00,1277,221.1898,8.203896,0.0,16.407792,0.0,0.0363,40.844443,20.266280,ALB,2493.9715,452.0053,1,6,145,1,5.399021
1,2012-01-01 00:00:00+00:00,1278,290.9656,3.410292,0.0,6.820585,0.0,0.0406,41.419464,19.859433,ALB,927.3392,167.9947,1,6,145,1,5.673205
2,2012-01-01 00:00:00+00:00,1279,19.3036,17.774111,0.0,35.548222,0.0,0.2352,41.702341,19.610791,ALB,187.6289,151.1404,1,6,145,1,2.960292
3,2012-01-01 00:00:00+00:00,1301,66.5712,28.651808,0.0,57.303615,0.0,0.1164,41.030359,19.159265,ALB,501.3857,492.2991,1,6,145,1,4.198272
4,2012-01-01 00:00:00+00:00,1302,41.3080,20.062614,0.0,40.125228,0.0,0.1687,41.282682,19.303039,ALB,197.8724,237.8496,1,6,145,1,3.721056


In [11]:
t_max = feat["Time"].max()
t_min = feat["Time"].min()

# Anchor windows off the extremes so this works for 24- or 36-month datasets.
val_start   = t_max - pd.DateOffset(months=6)
test_start  = t_max - pd.DateOffset(months=12)
train_end   = t_min + pd.DateOffset(months=12)

def label_split(ts):
    if ts < train_end:
        return "train"
    if test_start <= ts < val_start:
        return "test"
    if ts >= val_start:
        return "val"
    return "unused"

feat["split"] = feat["Time"].map(label_split)
print(feat.groupby("split")["Time"].agg(["min", "max", "count"]))

                             min                       max     count
split                                                               
test   2013-12-31 23:00:00+00:00 2014-06-30 22:00:00+00:00   6489936
train  2012-01-01 00:00:00+00:00 2012-12-31 23:00:00+00:00  13123296
unused 2013-01-01 00:00:00+00:00 2013-12-31 22:00:00+00:00  13085946
val    2014-06-30 23:00:00+00:00 2014-12-31 23:00:00+00:00   6598998


In [ ]:
FORMULA = "C(hour_of_week) + C(month)"

def rmse(y, yhat):
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def mape(y, yhat):
    return float(np.mean(np.abs((y - yhat) / y)) * 100)

results = {}
metrics_rows = []

for country, sub in feat.groupby("country", sort=False):
    train = sub[sub["split"] == "train"]
    test  = sub[sub["split"] == "test"]
    val   = sub[sub["split"] == "val"]
    if len(train) < 24 * 30 or len(test) == 0 or len(val) == 0:
        print(f"[skip] {country}: insufficient data")
        continue

    # Design matrices — build on train, align test/val to same columns
    X_train = dmatrix(FORMULA, data=train, return_type="dataframe")
    X_test  = dmatrix(FORMULA, data=test,  return_type="dataframe").reindex(
                  columns=X_train.columns, fill_value=0)
    X_val   = dmatrix(FORMULA, data=val,   return_type="dataframe").reindex(
                  columns=X_train.columns, fill_value=0)

    model = sm.OLS(train["log_demand"].values, X_train).fit()

    # Predict in log space, exponentiate back to MWh
    yhat_train = np.exp(model.predict(X_train))
    yhat_test  = np.exp(model.predict(X_test))
    yhat_val   = np.exp(model.predict(X_val))

    results[country] = {
        "model": model,
        "train": train.assign(yhat=yhat_train.values),
        "test":  test.assign(yhat=yhat_test.values),
        "val":   val.assign(yhat=yhat_val.values),
    }

    for name, part, yhat in [
        ("train", train, yhat_train),
        ("test",  test,  yhat_test),
        ("val",   val,   yhat_val),
    ]:
        y = part["demand_MWh"].values
        metrics_rows.append({
            "country": country, "split": name,
            "rmse_mwh": rmse(y, yhat), "mape_pct": mape(y, yhat),
            "n": len(y),
        })

metrics = pd.DataFrame(metrics_rows)
metrics_wide = metrics.pivot(index="country", columns="split",
                             values=["rmse_mwh", "mape_pct"])
print(metrics_wide.round(2))

In [ ]:
country_to_plot = next(iter(results))  # or set manually, e.g. "DE"
r = results[country_to_plot]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharey=True)

# Two weeks from test set
sample_test = r["test"].head(24 * 14)
axes[0].plot(sample_test["timestamp"], sample_test["demand_mwh"], label="actual", lw=1)
axes[0].plot(sample_test["timestamp"], sample_test["yhat"], label="forecast", lw=1)
axes[0].set_title(f"{country_to_plot} — test (first 2 weeks)")
axes[0].legend()

# Two weeks from validation
sample_val = r["val"].head(24 * 14)
axes[1].plot(sample_val["timestamp"], sample_val["demand_mwh"], label="actual", lw=1)
axes[1].plot(sample_val["timestamp"], sample_val["yhat"], label="forecast", lw=1)
axes[1].set_title(f"{country_to_plot} — validation (first 2 weeks)")
axes[1].legend()

plt.tight_layout()
plt.show()